# TransferQueue Tutorial — Basic Usage

This notebook walks through the core **Key-Value (KV) interface** of
[TransferQueue](https://github.com/Ascend/TransferQueue), an asynchronous
streaming data management module for efficient post-training workflows.

**What you will learn:**

1. Initialise TransferQueue (with Ray)
2. Store a single sample — `kv_put`
3. Store a batch of samples — `kv_batch_put`
4. Retrieve data — `kv_batch_get`
5. List stored keys & tags — `kv_list`
6. Partial-key and partial-field retrieval
7. Add fields, merge response chunks and clear selected fields
8. Working with nested (variable-length) tensors
9. Storing variable-size image data
10. Storing non-tensor data (`NonTensorStack`)
11. Load files on put and select tensor rows on get
12. Multiple partitions
13. Share fields across samples
14. Coordinate operations with local and global KV locks
15. Save and restore selected samples
16. Reclaim data and close TransferQueue

**Prerequisites:** Install this checkout with `pip install -e .` from the repository
root and select a notebook kernel using that environment. The examples use CPU
PyTorch and SimpleStorage.

Restart the kernel and run the cells in order. The initialization cell enables
global locks before starting TQ so the later lock examples can use them.

## 1. Initialization

TransferQueue runs on top of [Ray](https://www.ray.io/).  
We start Ray, then call `tq.init()` with a minimal configuration that uses the
built-in **SimpleStorage** backend.

In [ ]:
import os

import ray
import torch
from omegaconf import OmegaConf
from tensordict import TensorDict
from tensordict.tensorclass import NonTensorStack

import transfer_queue as tq

os.environ["TQ_ENABLE_KV_GLOBAL_LOCK"] = "1"
ray.init(ignore_reinit_error=True, include_dashboard=False)

config = OmegaConf.create(
    {
        "controller": {"polling_mode": True},
        "backend": {
            "storage_backend": "SimpleStorage",
            "SimpleStorage": {
                "total_storage_size": 200,
                "num_data_storage_units": 2,
            },
        },
    }
)

tq.init(config)
print("TransferQueue is ready!")

TransferQueue is ready!


## 2. Store a Single Sample — `kv_put`

`kv_put` stores **one** key-value pair.  
- `key` — a unique string identifier for the sample  
- `partition_id` — a logical namespace (like a table name)  
- `fields` — a `dict` of tensors **or** a `TensorDict`  
- `tag` — optional metadata dict attached to the key

In [ ]:
tq.kv_put(
    key="sample_0",
    partition_id="train",
    fields={"input_ids": torch.tensor([1, 2, 3, 4])},
    tag={"source": "wikipedia", "score": 0.95},
)
print("Stored sample_0")

Stored sample_0


You can also pass a pre-built `TensorDict` directly (the batch dimension
must be 1 for `kv_put`):

In [ ]:
fields_td = TensorDict(
    {
        "input_ids": torch.tensor([[5, 6, 7, 8]]),
        "attention_mask": torch.ones(1, 4, dtype=torch.long),
    },
    batch_size=1,
)

tq.kv_put(
    key="sample_1",
    partition_id="train",
    fields=fields_td,
    tag={"source": "books", "score": 0.88},
)
print("Stored sample_1")

Stored sample_1


## 3. Store a Batch of Samples — `kv_batch_put`

When you have multiple samples, `kv_batch_put` is more efficient than
calling `kv_put` in a loop.  The `fields` TensorDict must have
`batch_size == len(keys)`.

In [ ]:
keys = ["batch_0", "batch_1", "batch_2"]

fields = TensorDict(
    {
        "input_ids": torch.tensor([[10, 20], [30, 40], [50, 60]]),
        "attention_mask": torch.ones(3, 2, dtype=torch.long),
    },
    batch_size=3,
)

tags = [
    {"split": "train", "idx": 0},
    {"split": "train", "idx": 1},
    {"split": "train", "idx": 2},
]

tq.kv_batch_put(keys=keys, partition_id="train", fields=fields, tags=tags)
print(f"Stored {len(keys)} samples in one call")

Stored 3 samples in one call


## 4. Retrieve Data — `kv_batch_get`

Retrieve samples by key(s).  The result is always a `TensorDict`.

In [ ]:
# Retrieve a single key (pass a string)
result = tq.kv_batch_get(keys="sample_0", partition_id="train")
print("sample_0 →", result)
print("input_ids:", result["input_ids"])

sample_0 → TensorDict(batch_size=[1], fields={input_ids: i64[j1]})
input_ids: NestedTensor(size=(1, j1), offsets=tensor([0, 4]), contiguous=True)


In [ ]:
# Retrieve multiple keys at once
result = tq.kv_batch_get(keys=keys, partition_id="train")
print("batch result →", result)
print("input_ids:\n", result["input_ids"])
print("attention_mask:\n", result["attention_mask"])

batch result → TensorDict(batch_size=[3], fields={attention_mask: i64[j2], input_ids: i64[j3]})
input_ids:
 NestedTensor(size=(3, j3), offsets=tensor([0, 2, 4, 6]), contiguous=True)
attention_mask:
 NestedTensor(size=(3, j2), offsets=tensor([0, 2, 4, 6]), contiguous=True)


## 5. List Keys & Tags — `kv_list`

`kv_list` returns a nested dict:
```
{ partition_id: { key: tag_dict, ... }, ... }
```
Pass `partition_id` to filter, or omit it to see everything.

In [ ]:
info = tq.kv_list(partition_id="train")

for partition, key_tags in info.items():
    print(f"\nPartition: {partition}")
    for key, tag in key_tags.items():
        print(f"  {key}: {tag}")


Partition: train
  sample_0: {'source': 'wikipedia', 'score': 0.95}
  sample_1: {'source': 'books', 'score': 0.88}
  batch_0: {'split': 'train', 'idx': 0}
  batch_1: {'split': 'train', 'idx': 1}
  batch_2: {'split': 'train', 'idx': 2}


## 6. Partial-Key and Partial-Field Retrieval

You don't have to retrieve *all* keys or *all* fields at once.

### 6a. Partial Keys

Just pass a subset of the keys you stored.

In [ ]:
partial = tq.kv_batch_get(keys=["batch_0", "batch_2"], partition_id="train")
print("Partial-key input_ids:\n", partial["input_ids"])
assert partial["input_ids"].shape[0] == 2  # only 2 rows

Partial-key input_ids:
 NestedTensor(size=(2, j5), offsets=tensor([0, 2, 4]), contiguous=True)


### 6b. Partial Fields

Use `select_fields` to select columns. Every requested key must exist, and every
explicitly selected field must be ready in every key; otherwise the read raises
`ValueError`. Without `select_fields`, a batch read returns only fields ready in
all requested keys. Use `kv_exists` to probe optional keys without an error.

In [ ]:
# Retrieve only input_ids (single field)
result = tq.kv_batch_get(keys="sample_1", partition_id="train", select_fields="input_ids")
print("Fields returned:", list(result.keys()))
assert "input_ids" in result.keys()
assert "attention_mask" not in result.keys()

# Retrieve a specific set of fields
result = tq.kv_batch_get(
    keys="sample_1",
    partition_id="train",
    select_fields=["input_ids", "attention_mask"],
)
print("Fields returned:", list(result.keys()))

Fields returned: ['input_ids']
Fields returned: ['attention_mask', 'input_ids']


## 7. Adding and Updating Fields

### 7a. Add a new field

`kv_put` / `kv_batch_put` add or replace the fields supplied in the call, preserving
the other fields. Tags hold application metadata; TQ tracks field readiness
separately from those tags.

In [ ]:
# Add a "response" field to the batch keys
response_fields = TensorDict(
    {"response": torch.tensor([[100, 200], [300, 400], [500, 600]])},
    batch_size=3,
)

tq.kv_batch_put(keys=keys, partition_id="train", fields=response_fields)

result = tq.kv_batch_get(keys=keys, partition_id="train")
print("All fields now:", list(result.keys()))
print("response:\n", result["response"])

All fields now: ['attention_mask', 'input_ids', 'response']
response:
 NestedTensor(size=(3, j11), offsets=tensor([0, 2, 4, 6]), contiguous=True)


### 7b. Build a response in chunks

`kv_update` merges new values into a stored field with `merge_fn(old, new)`.
A missing field takes the first value directly. With `finalize=False`, the field
stays unready until the final update publishes it with `finalize=True`.

`kv_describe` shows stored and unfinalized fields; `kv_exists` reports ready fields.
This example uses two fixed token chunks to illustrate the updates. See
[Tutorial 02](02_kv_interface.py) for the KV interface walkthrough.

In [ ]:
streaming_key = "streaming_sample"
tq.kv_put(streaming_key, "train", tag={"status": "running"})

print("[Step 1] Storing the first response chunk with finalize=False...")
tq.kv_update(
    streaming_key,
    "train",
    fields={"response": torch.tensor([10, 11])},
    merge_fn=lambda old, new: torch.cat([old, new]),
    finalize=False,
)
print("  Stored fields:", tq.kv_describe(streaming_key, "train")[streaming_key])
print("  Ready fields:", tq.kv_exists(streaming_key, "train")[streaming_key])

print("\n[Step 2] Appending the final chunk and publishing the response...")
tq.kv_update(
    streaming_key,
    "train",
    fields={"response": torch.tensor([12, 13])},
    merge_fn=lambda old, new: torch.cat([old, new]),
    finalize=True,
)
complete = tq.kv_batch_get(streaming_key, "train", select_fields="response")
assert complete["response"][0].tolist() == [10, 11, 12, 13]
print("  Published response:", complete["response"][0].tolist())
tq.kv_put(streaming_key, "train", tag={"status": "finish"})

[Step 1] Storing the first response chunk with finalize=False...
  Stored fields: {'fields': ['response'], 'unfinalized': ['response'], 'links': {}}
  Ready fields: []

[Step 2] Appending the final chunk and publishing the response...
  Published response: [10, 11, 12, 13]


KVBatchMeta(keys=['streaming_sample'], tags=[{'status': 'finish'}], partition_id='train', fields=['response'], extra_info={})

Incremental updates require SimpleStorage. Use one writer per key or coordinate
writers with a KV lock. A timed-out merge may have committed, so retrying it
blindly can append the same chunk twice.

### 7c. Clear selected fields while keeping keys

The batch samples from section 3 still have `attention_mask`. Clearing that field
keeps their other fields, tags and consumption state.

In [ ]:
print("Clearing attention_mask from the batch samples...")
tq.kv_clear(keys, "train", fields="attention_mask")
print("  Keys and remaining fields:", tq.kv_exists(keys, "train"))
tags_by_key = tq.kv_list("train")["train"]
print("  Tags:", {key: tags_by_key[key] for key in keys})

Clearing attention_mask from the batch samples...
  Keys and remaining fields: {'batch_0': ['input_ids', 'response'], 'batch_1': ['input_ids', 'response'], 'batch_2': ['input_ids', 'response']}
  Tags: {'batch_0': {'split': 'train', 'idx': 0}, 'batch_1': {'split': 'train', 'idx': 1}, 'batch_2': {'split': 'train', 'idx': 2}}


## 8. Working with Nested (Variable-Length) Tensors

In many NLP and RL workloads each sample has a **different sequence length**
(e.g. generated responses).  PyTorch represents these as
[nested tensors](https://pytorch.org/docs/stable/nested.html) with the
**jagged layout** (`layout=torch.jagged`), and TransferQueue handles them
natively.

> **Note:** Because individual samples have different shapes, you must use
> `kv_batch_put` (not `kv_put`) to store nested tensors.

In [ ]:
nested_keys = ["nested_0", "nested_1", "nested_2"]

# Each sample has a different sequence length
nested_responses = torch.nested.as_nested_tensor(
    [
        torch.tensor([10, 11, 12]),  # length 3
        torch.tensor([20]),  # length 1
        torch.tensor([30, 31]),  # length 2
    ],
    layout=torch.jagged,
)

nested_fields = TensorDict(
    {
        "input_ids": torch.tensor([[1, 2], [3, 4], [5, 6]]),
        "response": nested_responses,
    },
    batch_size=3,
)

tq.kv_batch_put(
    keys=nested_keys,
    partition_id="train",
    fields=nested_fields,
    tags=[{"len": 3}, {"len": 1}, {"len": 2}],
)
print("Stored 3 samples with variable-length responses")

Stored 3 samples with variable-length responses


In [ ]:
# Retrieve all nested samples
result = tq.kv_batch_get(keys=nested_keys, partition_id="train")
print("input_ids:\n", result["input_ids"])
print("\nresponse (nested tensor):")
for i, sample in enumerate(result["response"]):
    print(f"  sample {i}: {sample}  (length {sample.shape[0]})")

input_ids:
 NestedTensor(size=(3, j14), offsets=tensor([0, 2, 4, 6]), contiguous=True)

response (nested tensor):
  sample 0: tensor([10, 11, 12])  (length 3)
  sample 1: tensor([20])  (length 1)
  sample 2: tensor([30, 31])  (length 2)


Partial-key retrieval works the same way with nested tensors — only the
requested samples are returned:

In [ ]:
# Retrieve only the first and last sample
partial = tq.kv_batch_get(keys=["nested_0", "nested_2"], partition_id="train")
print("Partial-key input_ids:\n", partial["input_ids"])
print("\nPartial-key responses:")
for i, sample in enumerate(partial["response"]):
    print(f"  sample {i}: {sample}")

# Verify correctness
assert torch.equal(partial["response"][0], torch.tensor([10, 11, 12]))
assert torch.equal(partial["response"][1], torch.tensor([30, 31]))
print("\nAssertions passed!")

Partial-key input_ids:
 NestedTensor(size=(2, j16), offsets=tensor([0, 2, 4]), contiguous=True)

Partial-key responses:
  sample 0: tensor([10, 11, 12])
  sample 1: tensor([30, 31])

Assertions passed!


Higher-dimensional nested tensors work too. Here each sample is a 3D
tensor with a variable first dimension (e.g. a different number of
attention heads or generated candidates):

In [ ]:
nested_3d_keys = ["nd3d_0", "nd3d_1", "nd3d_2"]

nested_3d = torch.nested.as_nested_tensor(
    [
        torch.randn(2, 3, 4),  # 2 heads
        torch.randn(5, 3, 4),  # 5 heads
        torch.randn(1, 3, 4),  # 1 head
    ],
    layout=torch.jagged,
)

fields_3d = TensorDict(
    {
        "input_ids": torch.tensor([[1, 2], [3, 4], [5, 6]]),
        "hidden_states": nested_3d,
    },
    batch_size=3,
)

tq.kv_batch_put(keys=nested_3d_keys, partition_id="train", fields=fields_3d)

result_3d = tq.kv_batch_get(keys=nested_3d_keys, partition_id="train")
for i, sample in enumerate(result_3d["hidden_states"]):
    print(f"sample {i} hidden_states shape: {sample.shape}")

# Clean up nested-tensor keys
tq.kv_clear(keys=nested_keys, partition_id="train")
tq.kv_clear(keys=nested_3d_keys, partition_id="train")

sample 0 hidden_states shape: torch.Size([2, 3, 4])
sample 1 hidden_states shape: torch.Size([5, 3, 4])
sample 2 hidden_states shape: torch.Size([1, 3, 4])


## 9. Storing Variable-Size Image Data

A common multimodal scenario: each sample in a batch contains a
**different number of images**, and each image has a **different
resolution**.  We can model this with a list of nested tensors — one
nested tensor per sample — wrapped inside a `TensorDict`.

Since the data is doubly ragged (variable count *and* variable size),
we store each image as a flattened 1-D tensor and pack all images per
sample into a single jagged nested tensor.  This way every sample is
one element of the batch, yet images retain their individual sizes.

In [ ]:
image_keys = ["img_sample_0", "img_sample_1", "img_sample_2"]

# Sample 0: 2 images — 3×32×32 (RGB, 32×32) and 3×64×64
sample_0_images = [torch.randn(3, 32, 32), torch.randn(3, 64, 64)]

# Sample 1: 1 image — 3×48×48
sample_1_images = [torch.randn(3, 48, 48)]

# Sample 2: 3 images — 3×16×16, 3×24×24, 3×32×64
sample_2_images = [torch.randn(3, 16, 16), torch.randn(3, 24, 24), torch.randn(3, 32, 64)]


# Flatten each image to 1-D so they can live in a single jagged nested tensor per sample
def flatten_images(images):
    return torch.cat([img.flatten() for img in images])


pixel_data = torch.nested.as_nested_tensor(
    [
        flatten_images(sample_0_images),  # 3*32*32 + 3*64*64 = 15360
        flatten_images(sample_1_images),  # 3*48*48            = 6912
        flatten_images(sample_2_images),
    ],  # 3*16*16 + 3*24*24 + 3*32*64 = 8736
    layout=torch.jagged,
)

# Store the number of pixels per image so we can reconstruct later
image_shapes = torch.nested.as_nested_tensor(
    [
        torch.tensor([[3, 32, 32], [3, 64, 64]]),  # 2 images
        torch.tensor([[3, 48, 48]]),  # 1 image
        torch.tensor([[3, 16, 16], [3, 24, 24], [3, 32, 64]]),  # 3 images
    ],
    layout=torch.jagged,
)

fields_img = TensorDict(
    {
        "prompt": torch.tensor([[101, 102], [201, 202], [301, 302]]),
        "pixel_data": pixel_data,
        "image_shapes": image_shapes,
    },
    batch_size=3,
)

tags_img = [
    {"num_images": 2},
    {"num_images": 1},
    {"num_images": 3},
]

tq.kv_batch_put(keys=image_keys, partition_id="train", fields=fields_img, tags=tags_img)
print("Stored 3 samples with variable numbers of variable-size images")

Stored 3 samples with variable numbers of variable-size images


In [ ]:
# Retrieve and reconstruct the images
result_img = tq.kv_batch_get(keys=image_keys, partition_id="train")

for i in range(result_img.batch_size[0]):
    shapes = result_img["image_shapes"][i]  # (num_images, 3) tensor
    pixels = result_img["pixel_data"][i]  # flat 1-D tensor of all pixels
    num_images = shapes.shape[0]

    offset = 0
    reconstructed = []
    for j in range(num_images):
        c, h, w = shapes[j].tolist()
        numel = c * h * w
        img = pixels[offset : offset + numel].reshape(c, h, w)
        reconstructed.append(img)
        offset += numel

    print(f"Sample {i}: {num_images} image(s) → {[tuple(img.shape) for img in reconstructed]}")

# Clean up
tq.kv_clear(keys=image_keys, partition_id="train")

Sample 0: 2 image(s) → [(3, 32, 32), (3, 64, 64)]
Sample 1: 1 image(s) → [(3, 48, 48)]
Sample 2: 3 image(s) → [(3, 16, 16), (3, 24, 24), (3, 32, 64)]


## 10. Storing Non-Tensor Data — `NonTensorStack`

Not every field is a numeric tensor.  Prompts, file paths, JSON metadata,
or arbitrary Python objects can be stored as **non-tensor data** using
tensordict's `NonTensorStack`.

- `NonTensorStack` wraps a **batch** of Python objects — one per sample.

TransferQueue serialises them transparently alongside regular tensors.

In [ ]:
nt_keys = ["nt_sample_0", "nt_sample_1", "nt_sample_2"]

# Build a TensorDict that mixes tensors with non-tensor data
nt_fields = TensorDict(
    {
        "input_ids": torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]]),
        # NonTensorStack: one string per sample in the batch
        "prompt_text": NonTensorStack(
            "Summarise the following article.",
            "Translate to French:",
            "Write a poem about rain.",
        ),
        # You can also store richer Python objects (dicts, lists, …)
        "metadata": NonTensorStack(
            {"source": "wiki", "lang": "en"},
            {"source": "books", "lang": "fr"},
            {"source": "user", "lang": "en"},
        ),
    },
    batch_size=3,
)

tq.kv_batch_put(keys=nt_keys, partition_id="train", fields=nt_fields)
print("Stored 3 samples with non-tensor fields")

Stored 3 samples with non-tensor fields


In [ ]:
# Retrieve and inspect non-tensor fields
result_nt = tq.kv_batch_get(keys=nt_keys, partition_id="train")

print("input_ids:\n", result_nt["input_ids"])

print("\nprompt_text (NonTensorStack):")
for i, text in enumerate(list(result_nt["prompt_text"])):
    print(f"  [{i}] {text!r}")

print("\nmetadata (NonTensorStack of dicts):")
for i, meta in enumerate(list(result_nt["metadata"])):
    print(f"  [{i}] {meta}")

# You can also add a NonTensorStack field to a single key via kv_put
tq.kv_put(
    key="single_nt",
    partition_id="train",
    fields={"label": NonTensorStack("positive"), "score": torch.tensor([0.99])},
)
single = tq.kv_batch_get(keys="single_nt", partition_id="train")
print(f"\nSingle-key non-tensor field: label={list(single['label'])}")

# Clean up
tq.kv_clear(keys=nt_keys, partition_id="train")
tq.kv_clear(keys="single_nt", partition_id="train")

input_ids:
 NestedTensor(size=(3, j26), offsets=tensor([0, 3, 6, 9]), contiguous=True)

prompt_text (NonTensorStack):
  [0] 'Summarise the following article.'
  [1] 'Translate to French:'
  [2] 'Write a poem about rain.'

metadata (NonTensorStack of dicts):
  [0] {'source': 'wiki', 'lang': 'en'}
  [1] {'source': 'books', 'lang': 'fr'}
  [2] {'source': 'user', 'lang': 'en'}

Single-key non-tensor field: label=[['positive']]


## 11. Parsing on the Storage Unit

### 11a. Load a file on put

A `data_parser` runs on SimpleStorage: the caller sends a file path, and storage
loads its tensor. The file must be accessible to storage until the put returns.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory


def load_features(data):
    data["features"] = [torch.load(path, map_location="cpu", weights_only=True) for path in data["features"]]
    return data


data_parser_keys = ["data_parser_sample_0"]
parser_partition = "parser"
with TemporaryDirectory(prefix="tq-notebook-parser-") as data_dir:
    file_path = Path(data_dir) / "features.pt"
    torch.save(torch.arange(12, dtype=torch.float32).reshape(3, 4), file_path)
    print("Loading features.pt on the storage unit...")
    tq.kv_batch_put(
        data_parser_keys,
        parser_partition,
        fields=TensorDict({"features": NonTensorStack(str(file_path))}, batch_size=1),
        data_parser=load_features,
    )

stored = tq.kv_batch_get(data_parser_keys, parser_partition)
print("Stored features:\n", stored["features"][0])

Loading features.pt on the storage unit...
Stored features:
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])


### 11b. Select rows on get

A get parser can select tensor rows before returning data to the caller.
`parser_args` supplies one argument per requested key. Parsers preserve field
names and sample order; get parsers must leave stored values unchanged.
See [Tutorial 08](08_data_parser.py) for batch examples and usage constraints.

In [ ]:
def select_rows(data, row_indexes):
    return {"features": [value[indexes] for value, indexes in zip(data["features"], row_indexes, strict=True)]}


selected = tq.kv_batch_get(
    data_parser_keys,
    parser_partition,
    select_fields="features",
    data_parser=select_rows,
    parser_args=[[0, 2]],
)
print("Rows 0 and 2 returned from storage:\n", selected["features"][0])

Rows 0 and 2 returned from storage:
 tensor([[ 0.,  1.,  2.,  3.],
        [ 8.,  9., 10., 11.]])


## 12. Multiple Partitions

Partitions provide logical isolation — the same key name can exist in
different partitions without conflict.

In [ ]:
tq.kv_put(
    key="val_sample_0",
    partition_id="validation",
    fields={"input_ids": torch.tensor([99, 98, 97])},
    tag={"split": "val"},
)

all_info = tq.kv_list()  # no partition_id → list everything
print("All partitions:", list(all_info.keys()))

All partitions: ['train', 'parser', 'validation']


## 13. Share Fields Across Samples

Store a field once in `tq.PUBLIC_PARTITION` and link several samples to it with
`kv_link`. This example shares a `features` tensor; each sample also gets its own
private `score` field. Public key names need an application-specific prefix
because applications on the same TQ share the public partition.

Shared fields require SimpleStorage. Linked fields and referenced public rows
cannot be overwritten. See [Tutorial 07](07_shared_fields.py) for sharing and
reclaiming public data, and [Tutorial 03](03_metadata_concepts.py) for `BatchMeta` links.

In [ ]:
print("[Step 1] Storing a shared field in the public partition...")
public_key = "notebook/shared_0"
shared_features = torch.arange(24, dtype=torch.float32).reshape(6, 4)
tq.kv_put(public_key, tq.PUBLIC_PARTITION, fields={"features": shared_features})
print(f"  Stored {public_key}: features shape={tuple(shared_features.shape)}")

print("\n[Step 2] Linking samples to the shared field...")
shared_keys = [f"shared_sample_{i}" for i in range(3)]
tq.kv_link(shared_keys, "shared", public_key=public_key, fields="features")
descriptions = tq.kv_describe(shared_keys, "shared")
for key in shared_keys:
    print(f"  {key} reads features from {descriptions[key]['links']['features']}")
print("  The links reference the public row; the tensor is stored once.")

[Step 1] Storing a shared field in the public partition...
  Stored notebook/shared_0: features shape=(6, 4)

[Step 2] Linking samples to the shared field...
  shared_sample_0 reads features from notebook/shared_0
  shared_sample_1 reads features from notebook/shared_0
  shared_sample_2 reads features from notebook/shared_0
  The links reference the public row; the tensor is stored once.


Reads combine linked fields with each sample's private fields. Here the three
samples share `features` and keep separate `score` values.

In [ ]:
print("[Step 3] Adding a private score to each sample...")
scores = torch.tensor([0.1, 0.2, 0.3])
tq.kv_batch_put(
    shared_keys,
    "shared",
    fields=TensorDict({"score": scores}, batch_size=len(shared_keys)),
)
print("  Each sample now has shared features and its own score.")

print("\n[Step 4] Reading shared and private fields together...")
shared = tq.kv_batch_get(shared_keys, "shared")
print("  Returned fields:", list(shared.keys()))
for i, key in enumerate(shared_keys):
    torch.testing.assert_close(shared["features"][i], shared_features)
    print(f"  {key}: features shape={tuple(shared['features'][i].shape)}, score={shared['score'][i].item():.1f}")

[Step 3] Adding a private score to each sample...
  Each sample now has shared features and its own score.

[Step 4] Reading shared and private fields together...


  Returned fields: ['features', 'score']
  shared_sample_0: features shape=(6, 4), score=0.1
  shared_sample_1: features shape=(6, 4), score=0.2
  shared_sample_2: features shape=(6, 4), score=0.3


## 14. Coordinate Operations with KV Locks

Local locks coordinate threads and async tasks within one process; global locks
coordinate processes across nodes in the same TQ cluster. Callers must use the
same kind of lock for the same keys; KV operations do not check locks automatically.

Use `async with` in a notebook. Global leases renew automatically; `lease.check()`
checks local lease state before writing and raises `LockLostError` if the lease
has expired or been lost. Global locks were enabled during initialization.
See [Tutorial 10](10_kv_locks.py) for supported operations and further details.

In [ ]:
tq.kv_put("local", "locks", fields={"count": torch.tensor(0)})
tq.kv_put("global", "locks", fields={"count": torch.tensor(0)})

print("Updating count under a local lock...")
async with tq.async_kv_local_lock("local", "locks", timeout=10):
    batch = await tq.async_kv_batch_get("local", "locks")
    count = int(batch["count"][0]) + 1
    await tq.async_kv_put("local", "locks", fields={"count": torch.tensor(count)})
print(f"Local lock released; count={count}")

Updating count under a local lock...
Local lock released; count=1


In [ ]:
print("Updating count under a global lock...")
async with tq.async_kv_global_lock("global", "locks", timeout=10) as lease:
    batch = await tq.async_kv_batch_get("global", "locks")
    count = int(batch["count"][0]) + 1
    lease.check()
    await tq.async_kv_put("global", "locks", fields={"count": torch.tensor(count)})
print(f"Global lock released; count={count}")

Updating count under a global lock...
Global lock released; count=1


## 15. Save and Restore Selected Samples

Save the existing `sample_0`, clear it, then restore its fields and tags.
For unfinished responses, shared fields and recovery after a restart, see
[Tutorial 09](09_data_dump.py).

In [ ]:
with TemporaryDirectory(prefix="tq-notebook-dump-") as dump_dir:
    dump_path = Path(dump_dir) / "sample"
    print("Saving sample_0...")
    tq.dump_data_by_key(dump_path, ["sample_0"], "train")
    tq.kv_clear("sample_0", "train")
    print("Cleared sample_0; restoring it from the dump...")
    restored_tags = tq.load_data_by_key(dump_path)

restored = tq.kv_batch_get("sample_0", "train")
print("Restored input_ids:", restored["input_ids"][0].tolist())
print("Restored tags:", restored_tags["sample_0"])

Saving sample_0...
Cleared sample_0; restoring it from the dump...
Restored input_ids: [1, 2, 3, 4]
Restored tags: {'source': 'wikipedia', 'score': 0.95}


## 16. Reclaim Data and Close

Clearing sample keys removes their links. Their public data remains until
`kv_clear_unlinked_public` reclaims the rows with no remaining links.

Each process closes its own client with `tq.close()`. The process that created TQ
also shuts down the shared services. `ray.shutdown()` closes the Ray connection.

In [ ]:
print("[Step 1] Clearing shared samples and reclaiming public data...")
tq.kv_clear(shared_keys, "shared")
print("  Cleared sample keys:", shared_keys)
print("  Public fields still available:", tq.kv_exists(public_key, tq.PUBLIC_PARTITION))
freed = tq.kv_clear_unlinked_public()
print("  Reclaimed public keys:", freed)

print("\n[Step 2] Clearing the other example keys...")
tq.kv_clear(["local", "global"], "locks")
tq.kv_clear(streaming_key, "train")
tq.kv_clear(keys="sample_0", partition_id="train")
tq.kv_clear(keys="sample_1", partition_id="train")
tq.kv_clear(keys=keys, partition_id="train")
tq.kv_clear(keys=data_parser_keys, partition_id=parser_partition)
tq.kv_clear(keys="val_sample_0", partition_id="validation")

remaining = tq.kv_list()
total_keys = sum(len(values) for values in remaining.values())
assert total_keys == 0
print(f"  Remaining keys across all partitions: {total_keys}")

[Step 1] Clearing shared samples and reclaiming public data...
  Cleared sample keys: ['shared_sample_0', 'shared_sample_1', 'shared_sample_2']
  Public fields still available: {'notebook/shared_0': ['features']}
  Reclaimed public keys: ['notebook/shared_0']

[Step 2] Clearing the other example keys...


  Remaining keys across all partitions: 0


In [ ]:
tq.close()
ray.shutdown()
print("TransferQueue and Ray shut down.")

TransferQueue and Ray shut down.


## API Reference for This Walkthrough

| Operation | Function | Notes |
|---|---|---|
| Initialize | `tq.init(config)` | Attached processes discover the existing TQ |
| Write | `kv_put`, `kv_batch_put` | Add or replace supplied fields and optionally merge tags |
| Merge | `kv_update`, `kv_batch_update` | SimpleStorage; keep chunks hidden with `finalize=False` |
| Read | `kv_batch_get(..., select_fields=...)` | Selected fields must be ready in every requested key |
| Parse on put/get | `data_parser`, get-side `parser_args` | SimpleStorage; preserve sample order and leave stored values unchanged on get |
| List / inspect | `kv_list`, `kv_describe`, `kv_exists` | Inspect tags, stored fields, links and readiness |
| Share | `kv_link`, `client.link` | SimpleStorage; link to fields in `PUBLIC_PARTITION` |
| Clear fields/keys | `kv_clear(..., fields=...)` | Omit `fields` to remove keys as well |
| Reclaim public rows | `kv_clear_unlinked_public` | Clear referencing samples first |
| Lock | `(async_)kv_local_lock`, `(async_)kv_global_lock` | Advisory; global locks must be enabled by the owner |
| Selective recovery | `dump_data_by_key`, `load_data_by_key` | SimpleStorage; restore public sources before linked rows |
| Close | `tq.close()` | Every process releases its own resources; only the owner shuts down TQ |

KV operations also have `async_` variants. Selective dump/load are synchronous.
For metadata-based access, see [Tutorial 03](03_metadata_concepts.py); for
sampler-driven consumption, see [Tutorial 06](06_streaming_dataloader.py).